# Multimodal_Lecture_Fusion 

In [16]:
# Future Ideas
"""
1) Chunk ASR text using sentence or puctuation aware chunking


"""

'\n1) Chunk ASR text using sentence or puctuation aware chunking\n\n\n'

In [12]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 3.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 1.4 MB/s eta 0:00:00


In [13]:
import os
import re
from PIL import Image, ImageEnhance, ImageOps
from pathlib import Path
from typing import List, Dict, Union
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

import docx
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

-------------
Methods from previous notebooks

In [14]:
def preprocess_image( image: Union[str, Path, np.ndarray, Image.Image], apply_grey: bool) -> Image.Image:
    """Light preprocessing (contrast boost) to improve OCR accuracy on both engines."""
    if isinstance(image, (str, Path)):
        img = Image.open(image).convert("RGB")
    elif isinstance(image, np.ndarray):
        img = Image.fromarray(image).convert("RGB")
    elif isinstance(image, Image.Image):
        img = image.convert("RGB")
    else:
        raise ValueError(f"Unsupported image type: {type(image)}")
    
    # resize first (important)
    img = img.resize((int(img.width * 1.5), int(img.height * 1.5)), Image.BICUBIC)

    if apply_grey:
        # convert to grayscale
        img = img.convert("L")
  
    # Boost contrast – helps especially with Arabic script and low-quality scans
    img = ImageOps.exif_transpose(img) # Reads EXIF tag Orientation
    enhancer = ImageEnhance.Contrast(img) 
    return enhancer.enhance(2)

------------------

In [15]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("Gemini")

In [16]:
import google.genai as genai
from google.genai.client import Client
# The client gets the API key from the environment variable `GEMINI_API_KEY`.
client = genai.Client(api_key=secret_value_0)

In [17]:
# Paths you must provide
ASR_TEXT = """تمام، واضح. سأقوم بإزالة الـ Terminator. طيب، الآن أريد أن أروي على الشكل التالي بالـ Ring Topology. لاحظ، هل ترون هذا الشكل الموجود معي هنا؟ الآن أريد أن أزيل هذا الـ Terminator مع هذا الـ Terminator، وأريد أن أربط النهاية بالبداية. صار عندي عبارة عن حلقة، وهذا نسميه Ring Topology. الرينج توبولوجي هذا هو الـ Ring Topology؛ لاحظ كل حاسب مرتبط بالحاسب الذي بعده، وكل حاسب مرتبط بالحاسب الذي بعده، والحاسب الأخير أين؟ لا يوجد حاسب أخير، الآن صاروا كلهم حواسب وراء بعض، لم تعد تفرق بين الحاسب الأول والحاسب الأخير، صارت عبارة عن حلقة، صح؟ طرحتم إيجابية من أول شيء، سلبية، خلوني أشوف سلبياتها. Okay، خلونا نشوف سلبياتها. الشبكة أكيد لها سلبيات؛ حاسب واحد إذا تعطل كيف يعمل؟ إذا تعطلت الشبكة أو تعطل حاسب، إذا تعطل أي واحد يطلع خارج الشبكة، صح؟ هل خرج خارجاً؟ تمام. طبعاً هذا بشكل عام، تفضل، الحاسب يكون في عبارات ما يعني ما نحكي عن التقنيات الموجودة، لا نحكي عن التقنيات، بل عن الشكل، لا نحكي فقط عن الصورة، تحكي عن تقنية تشكل الأنعام، نحكي فقط عن صورة؟ كذلك؟ تكلف، ظلوا على الإطار، ليس بشكل عام، هذه الشبكات، الـ Ring، هذه الشبكات بشكل عام تعمل فقط مع الـ Fiber Optic بشكل أساسي. ولما نحكي عن Fiber Optic، وتعرفون، الآن سننتقل على الـ Fiber Optic، أنا أحكي عن سرعات عالية وأحكي عن جودة الموضوع. على كل الأحوال، إذن هذا بشكل عام، هذه التوبولوجيا بشكل عام لها سلبيات ولها إيجابيات؛ من إيجابياتها السرعة العالية، ومن سلبياتها ما كنا نقوله: إذا واحد خرج خارجاً تطلع الشبكة كلها مفتوحة. موجودون عندكم هنا، هؤلاء اطلعوا عليهم فقط للاطلاع عليها. يجي عندي بعدين Mesh Topology، مش، شو يعني كلمة Mesh؟ الإنجليزية كلمة Mesh تعني فوضى. انظر ماذا عندي هنا، هذا هو الشكل، هذه التجهيزات الموجودة عندي، انظر شكلها، ماذا تقول عنها؟ ما هذه الفوضى؟ ما هكذا؟ صح؟ شو يعني هنا؟ أنت هنا عندك كل جهاز مرتبط مع كل الأجهزة، صح؟ طرحتم إيجابية فوراً، إيجابية من الصورة، راح أحس بـ .. كيف الكابل كثير يتكيف معها، إيجابيات راح أحس بـ .. في تصادمات بكل جهاز فيه تصادمات، سرعة، سرعة، سرعة.. أول شيء من أول عن الموضوع أنه هذه الشبكة لا يمكن توقف، صح؟ من الشكل حكايته، لأنه يبقى في جهاز معين حتى الجهاز الذي يعطل تلاحظ ممكن أنك أنت عن غير طريق أنك توصل للجهاز الذي تريد، يعني سلبية، تلاحظ تلاقي فيها عبارة عن كابلات. طبعاً هنا عندك إيجابيات وسلبيات تطلع عليها، ما في عبارة عن كابلات. في عندنا بعدين الـ Tree Topology، هذه الـ Tree Topology، لاحظ على الشكل هذا، في عندي الـ Tree، ما هي الـ Tree؟ هذا الجذع للأسفل والأغصان للأعلى، لاحظ هذا الشكل، لاحظ في عندي عبارة عن جذع Root ومرتبط لهون، وهذا مرتبط مع الـ Device هذا مع الـ Device هذا، وهذا مع اثنين وهذا مع اثنين، طلع عندي شكل شجرة. طيب، لماذا هذا؟ هي لماذا عملوا هذه؟ هي بشكل عام كمعايير وجدت بهذا الشكل لغاية أننا نحن مهندسي المعلوماتية عندما ندرس الشركات ونريد أن نعمل لهم عبارة عن شبكات وأنظمة برمجية، فنحن نحاول أن نجعل الشبكة تحاكي الهيكل الإداري، يعني هنا عندك مدير وهنا مدير وهنا قسم وهنا قسم، تعمل بهذا الشكل بحيث أنها بشكل عام تكون عندك شبكة تمثل كيف شكل الإدارة، أين المعلومات وأين المسؤول؟ أكثر معلومات تمنحونها للموظفين من أين مثلاً؟ ومن أين المعلومات تأتي وإلى أين؟ ومن أين تجمع المعلومات؟ ومن يعالجها؟ الحكايات التي نكتبها ونعكسها بالملخص تبع الشبكة. طبعاً، هناك Star Topology، وهناك Ring Topology، وكل واحدة منهم لها إيجابيات ولكل واحدة لها سلبيات. هل أنا يعني من الأفكار التي خطرت للناس الذين كانوا يعملون بالموضوع: هل أنا إذا جمعت Two Topologies مع بعض، ماذا سأستفيد؟ قالوا: ستستفيد إيجابياتها، لكنك ستواجه سلبياتها أيضاً، صح؟ هذا هو الواقع. فسلبيات الـ Ring وسلبيات الـ Bus وسلبيات الـ Star وسلبيات الـ Ring، تمام؟ بس الشبكة هذه بشكل عام أنا أعملها بحيث تحاكي بناءً إدارياً، يعني هي إدارية. تمام، Okay، في عندكم هنا سلبيات وإيجابيات موجودة، اطلعوا عليها. الآن هنا وضعت لكم Link للمحاضرة، هذه كل الكلام الذي حكيته مع الصور كلها موجودة باللينك أدناه."""   # paste the long text

ASR_TEXT = ASR_TEXT + "\nif we are going to talk about ancient civilization we have to talk about تدمر civilization"
IMAGES_TEXT = {'/kaggle/input/datasets/mahdiat11/wajeez-test-sample/Test_Samples/sample_1/images/0.png': {'text': 'Ring:\nEach device is connected to two neighbors, forming a closed loop. Data travels in one direction, and if one device fails, it can break the entire network.\n\nprepared by Dr. Imad M. Damascus university 2025/2026', 'visuals': []}, '/kaggle/input/datasets/mahdiat11/wajeez-test-sample/Test_Samples/sample_1/images/1.png': {'text': 'TOKEN RING\nSTATION\nSTATION\nSTATION\nSTATION\nDIRECTION OF FLOW\n\nprepared by Dr. Imad M. Damascus university 2025/2026', 'visuals': [{'coord': [70, 30, 840, 960], 'desc': 'مخطط يوضح طوبولوجيا الحلقة (Ring Topology) وبالتحديد تقنية (Token Ring)، حيث تتصل أربعة محطات (Stations) في شكل حلقة مغلقة، مع وجود أسهم زرقاء توضح اتجاه تدفق البيانات (Direction of Flow).'}]}, '/kaggle/input/datasets/mahdiat11/wajeez-test-sample/Test_Samples/sample_1/images/2.png': {'text': 'Node\nNode\nNode\nNode\nNode', 'visuals': [{'coord': [20, 70, 970, 930], 'desc': 'مخطط يوضح طوبولوجيا الشبكة الكاملة (Full Mesh Topology)، حيث ترتبط كل عقدة (Node) بجميع العقد الأخرى في الشبكة مباشرة عبر وصلات ثنائية الاتجاه (Bidirectional links) ممثلة بأسهم برتقالية.'}]}, '/kaggle/input/datasets/mahdiat11/wajeez-test-sample/Test_Samples/sample_1/images/3.jpg': {'text': 'سلبيات وإيجابيات ring topology\n\nإيجابيات:\n* نحتاج إلى كبل واحد فقط\n* تكلفة قليلة\n* ما من data collision\n* إضافة عقدة جديدة\n* لا يؤثر على ال bandwidth بسيطة\n\nسلبيات:\n* البيانات تحتاج أن تمر بكل جهاز بين المرسل والمستقبل\n* أي تعطل في الكبل يؤدي إلى توقف النظام كامل\n\nقانون عدد التوصيلات\nL = n(n-1) / 2\n\nسلبيات وإيجابيات mesh topology\n\nإيجابيات:\n* لا يوجد نقطة فشل واحدة\nNo single point of failure\n* أداء ممتاز\n* أمانه أفضل\n\nسلبيات:\n* تكلفة عالية جداً\n* غير عملية للشبكات الكبيرة\n* صيانة معقدة\n* تحتاج عدد كبير من المنافذ', 'visuals': [{'coord': [91, 84, 207, 363], 'desc': 'مخطط يوضح توبولوجيا الحلقة (Ring Topology) حيث تتصل الأجهزة في شكل دائري مغلق.'}, {'coord': [323, 0, 402, 188], 'desc': 'صندوق يحتوي على الصيغة الرياضية لحساب عدد الوصلات في توبولوجيا الشبكة (Network Connections Formula).'}, {'coord': [420, 91, 638, 698], 'desc': 'جدول مقارنة يوضح إيجابيات وسلبيات توبولوجيا الشبكة المتداخلة (Mesh Topology).'}]}, '/kaggle/input/datasets/mahdiat11/wajeez-test-sample/Test_Samples/sample_1/images/4.png': {'text': 'Tree Topology\nDevice 1\nDevice 2\nDevice 3\nDevice 4\nDevice 5\nDevice 6\nDevice 7\n\nprepared by Dr. Imad M. Damascus university 2025/2026', 'visuals': [{'coord': [158, 26, 950, 948], 'desc': 'مخطط توضيحي لتوبولوجيا الشجرة (Tree Topology) يظهر هيكلية هرمية تبدأ من Device 1 كجذر وتتفرع إلى بقية الأجهزة.'}]}, '/kaggle/input/datasets/mahdiat11/wajeez-test-sample/Test_Samples/sample_1/images/5.png': {'text': 'HYBRID NETWORK TOPOLOGY\nA hybrid topology is a type of network topology that combines two or more network topologies, including ring, bus, and mesh topologies\n\nzenarmor\nprepared by Dr. Imad M. Damascus university 2025/2026', 'visuals': [{'coord': [472, 78, 867, 938], 'desc': 'مخطط يوضح التوبولوجيا الهجينة (Hybrid Topology) التي تدمج بين توبولوجيا النجمة (Star)، والباص (Bus)، والحلقة (Ring) في شبكة واحدة.'}]}, '/kaggle/input/datasets/mahdiat11/wajeez-test-sample/Test_Samples/sample_1/images/6.png': {'text': 'Link for this lecture\n\nhttps://www.google.com/search?q=computer+net\nwork+topology&sca_esv=fb973f3e0fcee097&udm=\n2&biw=1360&bih=568&ei=YKjvaKPzIOK_i-\ngPkaaDoA0&ved=0ahUKEwijo-\nyXrqaQAxXi3wlHHRHTANQQ4dUDCBE&uact=5&oq\n=computer+network+topology&gs_lp=Egtnd3Mtd2l\n6LWltZyIZY29ntcHV0ZXIlgbmV0d29yayB0b3BvBG9ne\nUiD8wFQUg9Y4OUBcAJ4AJABAJgB8hygAdewAaoBA\nzktN7gBA8gBAPgBAZgCAKACAJgDAIgGAZIHAKAHu\nwKyBwC4BwDCBwDlBwA&sclient=gws-wiz-img', 'visuals': []}}

In [18]:
IMAGES_TEXT['/kaggle/input/datasets/mahdiat11/wajeez-test-sample/Test_Samples/sample_1/images/3.jpg']

{'text': 'سلبيات وإيجابيات ring topology\n\nإيجابيات:\n* نحتاج إلى كبل واحد فقط\n* تكلفة قليلة\n* ما من data collision\n* إضافة عقدة جديدة\n* لا يؤثر على ال bandwidth بسيطة\n\nسلبيات:\n* البيانات تحتاج أن تمر بكل جهاز بين المرسل والمستقبل\n* أي تعطل في الكبل يؤدي إلى توقف النظام كامل\n\nقانون عدد التوصيلات\nL = n(n-1) / 2\n\nسلبيات وإيجابيات mesh topology\n\nإيجابيات:\n* لا يوجد نقطة فشل واحدة\nNo single point of failure\n* أداء ممتاز\n* أمانه أفضل\n\nسلبيات:\n* تكلفة عالية جداً\n* غير عملية للشبكات الكبيرة\n* صيانة معقدة\n* تحتاج عدد كبير من المنافذ',
 'visuals': [{'coord': [91, 84, 207, 363],
   'desc': 'مخطط يوضح توبولوجيا الحلقة (Ring Topology) حيث تتصل الأجهزة في شكل دائري مغلق.'},
  {'coord': [323, 0, 402, 188],
   'desc': 'صندوق يحتوي على الصيغة الرياضية لحساب عدد الوصلات في توبولوجيا الشبكة (Network Connections Formula).'},
  {'coord': [420, 91, 638, 698],
   'desc': 'جدول مقارنة يوضح إيجابيات وسلبيات توبولوجيا الشبكة المتداخلة (Mesh Topology).'}]}

In [19]:
def chunk_text(text: str, chunk_size: int = 300) -> list[str]:
    """Basic sentence/punctuation-aware chunking."""
    sentences = re.split(r'(?<=[.؟!])\s+', text)
    chunks = []
    current_chunk = []
    current_len = 0
    
    for sentence in sentences:
        if current_len + len(sentence) > chunk_size and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_len = 0
        current_chunk.append(sentence)
        current_len += len(sentence)
        
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks

In [20]:
len(chunk_text(ASR_TEXT))

16

## Crop Illustrations

In [21]:
# 2. Crop Illustrations Function
def crop_illustrations(images_dict: dict, output_dir: str = "cropped_images") -> list:
    """
    Crops bounding boxes from images and returns an updated images_dict.

    images_dict: 
    {
        image_path: {
            'text': TEXT HERE,                                                                ADDED
            'visuals':[                                                                         ↓
                {'coord': image boudaries, 'desc':visualization description, 'path': the new cropped image for the visualiztion }
            ]
        }
    }
    """
    os.makedirs(output_dir, exist_ok=True)
    cropped_data = []
    
    for img_path, data in images_dict.items():
        if not os.path.exists(img_path):
            continue # Skip if file not found locally
            
        img = preprocess_image(img_path,apply_grey=False)
        W, H = img.size
        
        for idx, visual in enumerate(data.get('visuals', [])):
            ymin, xmin, ymax, xmax = visual['coord']
            
            # Convert normalized (0-1000) to pixel coordinates
            x1 = int((xmin / 1000) * W)
            y1 = int((ymin / 1000) * H)
            x2 = int((xmax / 1000) * W)
            y2 = int((ymax / 1000) * H)
            
            # Crop and save
            crop_img = img.crop((x1, y1, x2, y2))
            base_name = os.path.basename(img_path).split('.')[0]
            crop_filename = f"{base_name}_crop_{idx}.jpg"
            crop_path = os.path.join(output_dir, crop_filename)
            crop_img.save(crop_path)
            
            # update the dictionary wih crop_path
            visual["path"] = crop_path
            
    return images_dict

In [22]:
crop_illustrations(IMAGES_TEXT)

{'/kaggle/input/datasets/mahdiat11/wajeez-test-sample/Test_Samples/sample_1/images/0.png': {'text': 'Ring:\nEach device is connected to two neighbors, forming a closed loop. Data travels in one direction, and if one device fails, it can break the entire network.\n\nprepared by Dr. Imad M. Damascus university 2025/2026',
  'visuals': []},
 '/kaggle/input/datasets/mahdiat11/wajeez-test-sample/Test_Samples/sample_1/images/1.png': {'text': 'TOKEN RING\nSTATION\nSTATION\nSTATION\nSTATION\nDIRECTION OF FLOW\n\nprepared by Dr. Imad M. Damascus university 2025/2026',
  'visuals': [{'coord': [70, 30, 840, 960],
    'desc': 'مخطط يوضح طوبولوجيا الحلقة (Ring Topology) وبالتحديد تقنية (Token Ring)، حيث تتصل أربعة محطات (Stations) في شكل حلقة مغلقة، مع وجود أسهم زرقاء توضح اتجاه تدفق البيانات (Direction of Flow).',
    'path': 'cropped_images/1_crop_0.jpg'}]},
 '/kaggle/input/datasets/mahdiat11/wajeez-test-sample/Test_Samples/sample_1/images/2.png': {'text': 'Node\nNode\nNode\nNode\nNode',
  'visua

In [23]:
import shutil

folder_to_zip = "cropped_images"  # This is the output_dir from crop_illustrations

shutil.make_archive(folder_to_zip, 'zip', folder_to_zip)
print(f"Successfully zipped '{folder_to_zip}' into '{folder_to_zip}.zip'")

Successfully zipped 'cropped_images' into 'cropped_images.zip'


## Embeddings

In [24]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')  # better for Arabic + English

def get_embedding(text: str) -> np.ndarray:
    cleaned = re.sub(r'\s+', ' ', (text or '')).strip()
    vec = model.encode(cleaned, normalize_embeddings=True)
    return vec

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**future note** 

for speed you can process more than one chunck togather on gpu

texts = ["text1", "text2", "text3"]

embeddings = model.encode(texts, batch_size=32, normalize_embeddings=True)

In [25]:
get_embedding(chunk_text(ASR_TEXT)[0]).shape

(384,)

**Relying on apis here will lead to many errors so we need to have efficient fallbacks**

In [ ]:
# # ---------- Helpers ----------
# def _safe_stack(embs):
#     """Return (np.ndarray or None)."""
#     if not embs:
#         return None
#     return np.vstack(embs)

# def _cos_sim(a, b_mat):
#     """a: (d,) vector; b_mat: (n,d) matrix. returns (n,)"""
#     return cosine_similarity([a], b_mat)[0]

# def _best_idx_and_score(vec, mat):
#     sims = _cos_sim(vec, mat)
#     idx = int(np.argmax(sims))
#     return idx, float(sims[idx])

# def _build_collections(images_dict, crops):
#     """
#     Build per-slide/image "collections" of OCR + crops.
#     """
#     collections = []
#     crops_by_src = {}
#     for c in crops:
#         src = c.get("source_image_path")
#         crops_by_src.setdefault(src, []).append(c)
#     for image_name, image_data in images_dict.items():
#         collections.append({
#             "key": image_name,
#             "path": image_data.get("path"),
#             "ocr_text": (image_data.get("text") or "").strip(),
#             "crops": crops_by_src.get(image_name, [])
#         })
#     return collections

# def _collection_text_for_embedding(col):
#     """Text representing the collection's content for similarity matching."""
#     parts = []
#     if col.get("ocr_text"):
#         parts.append(col["ocr_text"])
#     for c in col.get("crops", []):
#         if c.get("desc"):
#             parts.append(c["desc"])
#     return "\n".join(parts).strip()

# def _is_novel_asr(asr_emb, collection_emb, novelty_threshold=0.78):
#     """If ASR is too similar to OCR/crops content, it is NOT novel."""
#     if collection_emb is None:
#         return True
#     sim = float(cosine_similarity([asr_emb], [collection_emb])[0][0])
#     return sim < novelty_threshold

# def set_paragraph_rtl(paragraph, align_right: bool = True):
#     """
#     Enforce true RTL direction (w:bidi) for Arabic paragraphs in Word.
#     Right-alignment alone is NOT enough.
#     """
#     pPr = paragraph._p.get_or_add_pPr()
#     # Add/ensure <w:bidi w:val="1"/>
#     bidi = pPr.find(qn("w:bidi"))
#     if bidi is None:
#         bidi = OxmlElement("w:bidi")
#         pPr.append(bidi)
#     bidi.set(qn("w:val"), "1")
#     if align_right:
#         # Add/ensure <w:jc w:val="right"/>
#         jc = pPr.find(qn("w:jc"))
#         if jc is None:
#             jc = OxmlElement("w:jc")
#             pPr.append(jc)
#         jc.set(qn("w:val"), "right")

# def _format_center(paragraph):
#     paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
#     return paragraph

# # ---------- Main ----------
# def create_lecture_docx(
#     asr_text: str,
#     images_dict: dict,
#     output_docx: str = "Lecture_Notes.docx",
#     *,
#     chunk_size: int = 300,
#     assign_threshold: float = 0.45,
#     novelty_threshold: float = 0.78,
#     max_asr_notes_per_collection: int = 6
# ):
#     """
#     Improved version:
#       - Keeps all original sophisticated logic (collections, embeddings, assignment, novelty filtering, no duplicate images).
#       - Adds proper RTL (bidi) support for Arabic.
#       - Uses LLM to intelligently blend OCR + crop descriptions + novel ASR into clean, formal, cohesive lecture notes.
#       - LLM output is strictly clean Arabic text only (no "Here is the answer", no markdown, no extra commentary).
#       - Crops/images are inserted visually first, followed by the LLM-generated explanation.
#     """
#     # Step 1: Chunk ASR
#     asr_chunks = chunk_text(asr_text, chunk_size=chunk_size)
#     print(f"Created {len(asr_chunks)} ASR chunks.")

#     # Step 2: Crop images and extract descriptions
#     crops = crop_illustrations(images_dict)
#     print(f"Cropped {len(crops)} illustrations.")

#     # Step 3: Build collections (OCR + crops grouped per slide/image)
#     collections = _build_collections(images_dict, crops)
#     print(f"Built {len(collections)} collections.")

#     # Step 4: Embed collections (primary source)
#     collection_texts = [_collection_text_for_embedding(c) for c in collections]
#     collection_embs = []
#     for t in collection_texts:
#         collection_embs.append(get_embedding(t if t else ""))
#     collection_embs_mat = _safe_stack(collection_embs)

#     # Step 5: Embed ASR chunks
#     print("Vectorizing ASR chunks...")
#     asr_embs = [get_embedding(ch) for ch in asr_chunks]

#     # Step 6: Assign ASR chunks to collections
#     assigned_asr = {i: [] for i in range(len(collections))}
#     if collection_embs_mat is not None and len(collections) > 0:
#         for idx, (chunk, emb) in enumerate(zip(asr_chunks, asr_embs)):
#             best_col_idx, score = _best_idx_and_score(emb, collection_embs_mat)
#             if score >= assign_threshold:
#                 assigned_asr[best_col_idx].append((score, idx, chunk, emb))
#         # Sort each collection's assigned ASR by best score first
#         for i in assigned_asr:
#             assigned_asr[i].sort(key=lambda x: x[0], reverse=True)

#     # ---------- LLM blending function (strict clean output) ----------
#     def generate_section(ocr: str, crops_list: list, novel_asr_chunks: list) -> str:
#         ocr_clean = ocr.strip() if ocr.strip() else "لا يوجد نص في الشريحة"
#         crops_text = "\n".join(
#             [f"- {c.get('desc', 'صورة توضيحية بدون وصف')}" for c in crops_list if c.get("desc")]
#         ) if crops_list else "لا توجد رسوم توضيحية"
#         asr_text_local = "\n".join(novel_asr_chunks) if novel_asr_chunks else "لا توجد ملاحظات صوتية إضافية"

#         prompt = f"""
# أنت تقوم بإنشاء ملخص محاضرة نظيف ومهني للطلاب باللغة العربية الفصحى.

# المحتوى الرئيسي من الشريحة (OCR):
# {ocr_clean}

# الشرح البصري من الرسومات المقصوصة:
# {crops_text}

# ملاحظات صوتية إضافية جديدة (غير مذكورة في الشريحة):
# {asr_text_local}

# تعليمات صارمة:
# - استخدم النص من الشريحة والصور التوضيحية كمصدر أساسي.
# - أضف فقط المعلومات الجديدة من الصوت إذا كانت مفيدة وغير مكررة.
# - أزل أي تكرار أو معلومات زائدة تماماً.
# - اكتب بشكل واضح، موجز، ومنظم (فقرات أو نقاط مرقمة إذا كان مناسباً).
# - استخدم اللغة العربية الفصحى واحتفظ بالمصطلحات الإنجليزية كما هي.
# - لا تضف أي مقدمات مثل "إليك الملخص" أو "النص الناتج" أو أي تعليقات إضافية.
# - لا تستخدم markdown أو علامات تنسيق أو نصوص إنجليزية زائدة.
# - الإخراج يجب أن يكون النص العربي النظيف فقط، جاهز للنسخ إلى وثيقة Word.

# الملخص النهائي:
# """

#         response = client.models.generate_content(
#             model="gemma-4-31b-it",
#             contents=prompt
#         )
#         parts = response.candidates[0].content.parts
#         texts = []
#         for part in parts:
#             if hasattr(part, "thought") and part.thought:
#                 continue
#             if hasattr(part, "text") and part.text:
#                 texts.append(part.text.strip())
#         return "\n\n".join(texts).strip()

#     # Step 7: Create Word Document
#     doc = Document()
#     style = doc.styles['Normal']
#     style.font.name = 'Arial'
#     style.font.size = Pt(14)

#     title = doc.add_heading('ملخص المحاضرة (Lecture Notes)', level=1)
#     title.alignment = WD_ALIGN_PARAGRAPH.CENTER

#     used_image_paths = set()

#     for i, col in enumerate(collections):
#         # Heading (RTL)
#         heading = doc.add_heading(f'الشريحة/القسم {i+1}: {col["key"]}', level=2)
#         set_paragraph_rtl(heading)

#         # Add crop images (visuals first - never repeat globally)
#         crops_list = col.get("crops", [])
#         if crops_list:
#             p_label = doc.add_paragraph("الرسومات/الصور التوضيحية:")
#             set_paragraph_rtl(p_label)
#             for c in crops_list:
#                 img_path = c.get("path")
#                 if not img_path or img_path in used_image_paths:
#                     continue
#                 used_image_paths.add(img_path)
#                 p_img = doc.add_paragraph()
#                 _format_center(p_img)
#                 run = p_img.add_run()
#                 run.add_picture(img_path, width=Inches(4.5))
#                 if c.get("desc"):
#                     cap = doc.add_paragraph(f"شرح توضيحي: {c['desc']}")
#                     _format_center(cap)
#                     set_paragraph_rtl(cap, align_right=False)  # bidi only, keep center
#                     for r in cap.runs:
#                         r.font.size = Pt(11)

#         # Prepare novel ASR chunks (already filtered by assignment + novelty)
#         novel_asr_chunks = []
#         cand = assigned_asr.get(i, [])
#         collection_emb = collection_embs[i] if i < len(collection_embs) else None
#         notes_added = 0
#         for score, asr_idx, chunk, emb in cand:
#             if notes_added >= max_asr_notes_per_collection:
#                 break
#             if collection_emb is not None and not _is_novel_asr(emb, collection_emb, novelty_threshold):
#                 continue
#             novel_asr_chunks.append(chunk)
#             notes_added += 1

#         # Generate clean blended notes with LLM
#         ocr_text = col.get("ocr_text", "")
#         final_text = generate_section(ocr_text, crops_list, novel_asr_chunks)

#         if final_text.strip():
#             p = doc.add_paragraph(final_text)
#             set_paragraph_rtl(p)
#         else:
#             p = doc.add_paragraph("لا يوجد محتوى إضافي لهذا القسم.")
#             set_paragraph_rtl(p)

#         doc.add_page_break()

#     doc.save(output_docx)
#     print(f"Success! Document saved to {output_docx}")

In [29]:
import re
import unicodedata
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

RLM = '\u200F'
LRM = '\u200E'

def _is_ltr_char(ch: str) -> bool:
    bidi = unicodedata.bidirectional(ch)
    return bidi in ('L', 'EN', 'ES', 'ET', 'AN')

def _is_arabic_char(ch: str) -> bool:
    try:
        return 'ARABIC' in unicodedata.name(ch)
    except ValueError:
        return False
        
def _split_runs(text: str):
    if not text:
        return []

    runs = []
    current = text[0]
    current_is_ar = _is_arabic_char(text[0])

    for ch in text[1:]:
        is_ar = _is_arabic_char(ch)
        if is_ar == current_is_ar:
            current += ch
        else:
            runs.append((current_is_ar, current))
            current = ch
            current_is_ar = is_ar

    runs.append((current_is_ar, current))
    return runs

def inject_bidi_marks(text: str) -> str:
    result = []
    for is_ar, segment in _split_runs(text):
        if is_ar:
             result.append(segment)
        else:
            stripped = segment.strip()
            lead = segment[:len(segment)-len(segment.lstrip())]
            trail = segment[len(segment.rstrip()):]
            result.append(f"{lead}{LRM}{stripped}{RLM}{trail}")

    return ''.join(result)

def add_rtl_paragraph(doc, text, font_name='Arial', font_size=Pt(12), alignment=WD_ALIGN_PARAGRAPH.RIGHT):
    fixed_text = inject_bidi_marks(text)
    p = doc.add_paragraph()
    run = p.add_run(fixed_text)
    p.alignment = alignment
    run.font.rtl = True
    run.font.name = font_name
    run.font.size = font_size
    return p

In [30]:
def _clean_text_for_embedding(text: str) -> str:
    if not text:
        return ""
    return re.sub(r'\s+', ' ', str(text)).strip()


def build_image_profiles(images_dict: dict) -> list[dict]:
    profiles = []
    for image_path, image_data in images_dict.items():
        visuals = list(image_data.get('visuals', []))
        ocr_text = _clean_text_for_embedding(image_data.get('text', ''))
        visual_texts = [
            _clean_text_for_embedding(visual.get('desc', ''))
            for visual in visuals
            if _clean_text_for_embedding(visual.get('desc', ''))
        ]
        visual_text = '\n'.join(visual_texts).strip()
        combined_text = '\n'.join(part for part in [ocr_text, visual_text] if part).strip()

        profiles.append({
            'image_path': image_path,
            'image_name': os.path.basename(image_path),
            'ocr_text': ocr_text,
            'visual_text': visual_text,
            'combined_text': combined_text,
            'visuals': visuals,
        })
    return profiles


def build_image_embeddings(images_dict: dict):
    image_profiles = build_image_profiles(images_dict)
    for profile in image_profiles:
        profile['ocr_embedding'] = get_embedding(profile['ocr_text']) if profile['ocr_text'] else None
        profile['visual_embedding'] = get_embedding(profile['visual_text']) if profile['visual_text'] else None
        profile['combined_embedding'] = get_embedding(profile['combined_text']) if profile['combined_text'] else None
    return image_profiles


def build_asr_embeddings(asr_text: str, chunk_size: int = 300):
    asr_chunks = chunk_text(asr_text, chunk_size=chunk_size)
    asr_embeddings = [get_embedding(_clean_text_for_embedding(chunk)) for chunk in asr_chunks]
    return asr_chunks, np.vstack(asr_embeddings) if asr_embeddings else None


def rank_chunks_for_image(image_profile, asr_embeddings, asr_chunks, top_k: int = 3, min_similarity: float = 0.35):
    if asr_embeddings is None or len(asr_chunks) == 0:
        return []

    score_vectors = []

    if image_profile.get('ocr_embedding') is not None:
        score_vectors.append((0.7, cosine_similarity([image_profile['ocr_embedding']], asr_embeddings)[0]))

    if image_profile.get('visual_embedding') is not None:
        score_vectors.append((0.3, cosine_similarity([image_profile['visual_embedding']], asr_embeddings)[0]))

    if not score_vectors and image_profile.get('combined_embedding') is not None:
        score_vectors.append((1.0, cosine_similarity([image_profile['combined_embedding']], asr_embeddings)[0]))

    if not score_vectors:
        return []

    scores = np.zeros(len(asr_chunks), dtype=float)
    for weight, vector in score_vectors:
        scores += weight * vector

    ranked_indices = np.argsort(scores)[::-1]
    top_chunks = []
    for idx in ranked_indices:
        score = float(scores[idx])
        if score < min_similarity:
            continue
        top_chunks.append((idx, asr_chunks[idx], score))
        if len(top_chunks) >= top_k:
            break
    return top_chunks

In [31]:
IMAGES_TEXT['/kaggle/input/datasets/mahdiat11/wajeez-test-sample/Test_Samples/sample_1/images/1.png']

{'text': 'TOKEN RING\nSTATION\nSTATION\nSTATION\nSTATION\nDIRECTION OF FLOW\n\nprepared by Dr. Imad M. Damascus university 2025/2026',
 'visuals': [{'coord': [70, 30, 840, 960],
   'desc': 'مخطط يوضح طوبولوجيا الحلقة (Ring Topology) وبالتحديد تقنية (Token Ring)، حيث تتصل أربعة محطات (Stations) في شكل حلقة مغلقة، مع وجود أسهم زرقاء توضح اتجاه تدفق البيانات (Direction of Flow).',
   'path': 'cropped_images/1_crop_0.jpg'}]}

In [32]:
def _find_matching_image(image_profile, asr_embeddings, asr_chunks, top_k: int = 3, min_similarity: float = 0.35):
    return rank_chunks_for_image(
        image_profile,
        asr_embeddings,
        asr_chunks,
        top_k=top_k,
        min_similarity=min_similarity,
    )

In [33]:
def _add_image_section(doc, image_item, top_chunks, used_crop_paths):
    heading = doc.add_heading(f'الصورة: {image_item["image_name"]}', level=2)
    heading.alignment = WD_ALIGN_PARAGRAPH.RIGHT

    if image_item.get('ocr_text'):
        ocr_p = add_rtl_paragraph(doc, f"النص المستخرج من الصورة: {image_item['ocr_text']}")
        ocr_p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        ocr_p.runs[0].font.size = Pt(11)

    if image_item.get('visual_text'):
        visual_text_paragraph = add_rtl_paragraph(doc, f"الوصف البصري المستخرج: {image_item['visual_text']}")
        visual_text_paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        visual_text_paragraph.runs[0].font.size = Pt(11)

    visuals = image_item.get('visuals', [])
    if visuals:
        visual_label = add_rtl_paragraph(doc, 'الرسومات التوضيحية المرتبطة بالصورة:')
        visual_label.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        visual_label.runs[0].font.size = Pt(11)

        for visual in visuals:
            crop_path = visual.get('path')
            desc = (visual.get('desc') or '').strip()

            if crop_path and crop_path in used_crop_paths:
                continue

            if crop_path and os.path.exists(crop_path):
                img_paragraph = doc.add_paragraph()
                img_paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
                img_run = img_paragraph.add_run()
                img_run.add_picture(crop_path, width=Inches(4.5))
                used_crop_paths.add(crop_path)

            if desc:
                desc_paragraph = add_rtl_paragraph(doc, f"شرح توضيحي: {desc}")
                desc_paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
                desc_paragraph.runs[0].font.size = Pt(10)

    chunk_heading = add_rtl_paragraph(doc, 'أقرب مقاطع ASR لهذه الصورة:')
    chunk_heading.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    chunk_heading.runs[0].font.size = Pt(11)

    if not top_chunks:
        empty_paragraph = add_rtl_paragraph(doc, 'لا توجد مقاطع مطابقة بدرجة كافية.')
        empty_paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        empty_paragraph.runs[0].font.size = Pt(11)
        return

    for rank, (_, chunk_text_value, score) in enumerate(top_chunks, start=1):
        chunk_paragraph = add_rtl_paragraph(
            doc,
            f"{rank}. [{score:.2f}] {chunk_text_value}",
        )
        chunk_paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        chunk_paragraph.runs[0].font.size = Pt(11)

In [34]:
def create_lecture_docx(
    asr_text: str,
    images_dict: dict,
    output_docx: str = "Lecture_Notes.docx",
    chunk_size: int = 300,
    top_k_chunks: int = 3,
    min_similarity: float = 0.35,
):
    asr_chunks, asr_embeddings = build_asr_embeddings(asr_text, chunk_size=chunk_size)
    image_profiles = build_image_embeddings(images_dict)

    print(f"Created {len(asr_chunks)} ASR chunks.")
    print(f"Built {len(image_profiles)} image profiles.")

    doc = Document()
    style = doc.styles['Normal']
    style.font.name = 'Arial'
    style.font.size = Pt(14)

    title = doc.add_heading('ملخص المحاضرة (Lecture Notes)', level=1)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER

    used_crop_paths = set()

    if len(image_profiles) == 0:
        p = add_rtl_paragraph(doc, 'لا توجد صور أو نصوص صور صالحة للمعالجة.')
        p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        doc.save(output_docx)
        print(f"Success! Document saved to {output_docx}")
        return

    for image_item in image_profiles:
        top_chunks = _find_matching_image(
            image_item,
            asr_embeddings,
            asr_chunks,
            top_k=top_k_chunks,
            min_similarity=min_similarity,
        )
        _add_image_section(doc, image_item, top_chunks, used_crop_paths)
        doc.add_page_break()

    doc.save(output_docx)
    print(f"Success! Document saved to {output_docx}")

In [35]:
import os
create_lecture_docx(ASR_TEXT, IMAGES_TEXT)

Created 16 ASR chunks.
Built 7 image profiles.
Success! Document saved to Lecture_Notes.docx
